# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIRˆ² colorectal cancer survivor dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, their `@id`s, and columns.

In [ ]:
# Get record sets in the dataset with their IDs
from pprint import pprint

# Most Croissant datasets use the `recordSet` attribute to declare data tables/record sets.
record_sets = list(dataset.record_sets)
print("Available record sets:")
for rs in record_sets:
    print(f"- name: {rs.name}\n  @id: {rs.id}")

# Explore the first record set fields (and columns if present)
if record_sets:
    main_rs = record_sets[0]
    print(f"\nFields in record set '{main_rs.name}' (@id: {main_rs.id}):")
    for field in main_rs.fields:
        print(f"  - field name: {field.name}")
        print(f"    @id: {field.id}")
        if hasattr(field, 'column') and field.column is not None:
            if isinstance(field.column, list):
                for col in field.column:
                    print(f"      column: {getattr(col, 'name', str(col))} @id: {getattr(col, 'id', str(col))}")
            else:
                print(f"      column: {getattr(field.column, 'name', str(field.column))} @id: {getattr(field.column, 'id', str(field.column))}")

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. All entities are referenced by their `@id`s.

In [ ]:
# Prepare to extract tabular data from the main record set

# Use the record set's @id for all data operations
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records from record set: {record_set_id}")

# Choose first record set for EDA demonstration
main_rs_id = record_set_ids[0]
print(f"\nColumns in main record set ({main_rs_id}):")
print(dataframes[main_rs_id].columns.tolist())

dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping/categorizing data. All fields are referenced by their `@id`.

In [ ]:
# Identify candidate numeric and categorical fields by @id
from mlcroissant.dataset.fields.field import Field

# Find a numeric field (e.g. age of subjects) for analysis
main_rs_obj = next((rs for rs in dataset.record_sets if rs.id == main_rs_id), None)

# Try to select a likely numeric field (e.g. Age or similar)
numeric_field_id = None
group_field_id = None
for f in main_rs_obj.fields:
    if hasattr(f, 'data_type') and f.data_type and ("Integer" in f.data_type or "Number" in f.data_type or "Float" in f.data_type):
        numeric_field_id = f.id
        print(f"Found numeric field: {f.name}, @id: {f.id}")
        break

# Pick a field for grouping (e.g. Sex or similar categorical)
for f in main_rs_obj.fields:
    if hasattr(f, 'data_type') and f.data_type and "Text" in f.data_type:
        if 'sex' in f.name.lower():
            group_field_id = f.id
            print(f"Found group field: {f.name}, @id: {f.id}")
            break

# For demonstration, fallback to the first columns if auto selection failed
if not numeric_field_id:
    numeric_field_id = dataframes[main_rs_id].columns[0]
if not group_field_id:
    group_field_id = dataframes[main_rs_id].columns[1]

df = dataframes[main_rs_id]

# EDA: filter, normalize, group
# First, make sure the numeric field is numeric
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = df[numeric_field_id].median() if df[numeric_field_id].notnull().any() else 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by the chosen categorical field, if it exists in the DataFrame
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field_id} (showing mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field after filtering
plt.figure(figsize=(8,5))
sns.histplot(filtered_df[numeric_field_id].dropna(), bins=15, color='steelblue')
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id} (filtered)")
plt.show()

# If group_field_id is available, plot boxplot by group
if group_field_id in filtered_df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIRˆ² dataset on clinicopathological and molecular characteristics of second primary colorectal cancer in cancer survivors using the `mlcroissant` library. We reviewed metadata, explored available record sets and their fields (referencing all entities by their `@id`), loaded the main record set to a DataFrame, performed standard exploratory data analysis, and visualized key numeric and categorical trends in the data.

Further analysis can consider advanced statistical modeling or use the record set and field `@id`s for reproducible data processing and workflow documentation.